# registry

> Defining hyperelements and collecting their styles and behaviors

In [ ]:
#| default_exp registry

A UI component needs HTML markup, CSS styling, and sometimes browser behavior. FastHTML already gives us the markup constructors (FastTags). Here we build a registry to hold the styling and behaviors, associate them with their constructor, and collect everything that a page needs into one stylesheet and one script.

We use the same registry system for non-component application styling (e.g. layout and spacing utilities etc.).

In [ ]:
import inspect
from fastcore.utils import *
import fasthtml.components

In [ ]:
from fasthtml.common import Label, Input, Button, to_xml
from fastcore.test import *

## Rules

Styling and behavior have the same shape. Each pairs a *selector* that picks elements with a *payload* saying what happens to them: CSS declarations for styling, an event and handler for behavior. Each must also reach the page inside one aggregated asset (one stylesheet, one script). So we register them together: a named entry holds CSS text and behavior records, filed under a [cascade layer](https://developer.mozilla.org/en-US/docs/Web/CSS/@layer).

The layer list is fixed and ordered, and each layer beats the ones before it regardless of specificity, so an application can restyle a built-in component without touching its definition. We use a plain dict for the registry, so registering a name again replaces the earlier entry.

In [ ]:
LAYERS = ('reset','tokens','base','layout','components','app','utilities')
_registry = {}

Behaviors are a dict mapping `(selector, event)` keys to JavaScript handler bodies. Each handler runs with `el` bound to the matched element and `e` to the event. A key may include a truthy third item to attach the handler in the capture phase, needed for events like `toggle` that don't bubble. An entry may also declare a root selector `sel`; `&` in a behavior selector then stands for it, as in CSS nesting:

In [ ]:
def _behaviors(on, sel):
    "Behavior records with `&` replaced by `sel`"
    res = []
    for (selector, event, *capture), js in (on or {}).items():
        if '&' in selector:
            if sel is None: raise ValueError(f'`&` in {selector!r} needs sel')
            selector = selector.replace('&', sel)
        res.append(dict(sel=selector, ev=event, js=js, cap=bool(capture and capture[0])))
    return res

`Rules` checks the layer, expands the behaviors, and registers itself.

In [ ]:
class Rules:
    def __init__(
        self,
        name,        # Registry key; registering it again replaces the earlier entry
        css='',      # CSS text, or a function returning it, called when the stylesheet is built
        on=None,     # Mapping of (selector, event[, capture]) to JavaScript handler bodies
        sel=None,    # Root selector; `&` in a behavior selector stands for it
        fn=None,     # Constructor emitting the markup these rules style
        layer='app', # One of LAYERS
        doc='',      # Description shown by vocab
    ):
        if layer not in LAYERS: raise ValueError(f'Unknown layer: {layer}')
        store_attr('name,css,sel,fn,layer,doc')
        self.on = _behaviors(on, sel)
        _registry[name] = self

    def __repr__(self):
        sel = f' {self.sel}' if self.sel else ''
        sig = f' {self.name}{inspect.signature(self.fn)}' if inspect.isfunction(self.fn) else ''
        doc = f': {self.doc}' if self.doc else ''
        return f'{self.name} [{self.layer}]{sel}{sig}{doc}'

The simplest entry is bare CSS with no constructor; for example, page structure like a centered column. There's no component to attach to; it's just CSS that belongs in the aggregated stylesheet. `app` is the default layer, since an application is the usual author of an entry like this:

In [ ]:
Rules('app-page', 'main { max-width: 65ch; margin-inline: auto; padding: 1rem; }', doc='A centered single-column page.')

app-page [app]: A centered single-column page.

A behavior with no constructor of its own works the same way. Here's a `copy` attribute: put it on any control, and a click copies the attribute's value, or failing that the element's text, to the clipboard. The attribute is the natural root selector, so we declare it as `sel` and write the behavior against `&`. What the entry stores is the expanded record:

In [ ]:
Rules('copy', sel='[copy]', on={('&', 'click'): r"navigator.clipboard.writeText(el.getAttribute('copy') || el.textContent)"},
    doc='A control carrying `copy` copies its value, or its text, to the clipboard.')
_registry['copy'].on

[{'sel': '[copy]',
  'ev': 'click',
  'js': "navigator.clipboard.writeText(el.getAttribute('copy') || el.textContent)",
  'cap': False}]

In [ ]:
with expect_fail(ValueError): Rules('app-links', 'a { color: teal; }', layer='appp')
with expect_fail(ValueError): Rules('paste', on={('&', 'click'): ''})

## Building the stylesheet

Now we can assemble the stylesheet. We start with a bare `@layer` statement listing the layer names to declare their precedence. We then emit each layer's entries as a block. Registry order decides source order within a group. An entry's CSS may be a function, called here, for rules that depend on registrations made after their own.

In [ ]:
def build_css():
    "Assemble the registered CSS in cascade-layer order"
    out = [f"@layer {', '.join(LAYERS)};"]
    for layer in LAYERS:
        items = [o for o in _registry.values() if o.layer==layer and o.css]
        if not items: continue
        body = '\n'.join(f"/* {o.name} */\n{(o.css() if callable(o.css) else o.css).strip()}" for o in items)
        out.append(f"@layer {layer} {{\n{body}\n}}")
    return '\n\n'.join(out) + '\n'

In [ ]:
print(build_css())

@layer reset, tokens, base, layout, components, app, utilities;

@layer app {
/* app-page */
main { max-width: 65ch; margin-inline: auto; padding: 1rem; }
}



## Registering a constructor

A component is rules plus a constructor (which could be either a standard FastTag or a python function bundling together multiple FastTags). `hypelt` registers the pair and returns the constructor.

In [ ]:
def hypelt(
    name=None,   # FastHTML tag name or a constructor; omit to use as a decorator
    *,
    sel=None,    # Root selector; required for a handwritten constructor
    css='',      # Ordinary CSS text
    on=None,     # Mapping of (selector, event[, capture]) to JavaScript handler bodies
    layer='app', # One of LAYERS; built-ins use components
    doc=None,    # Description; defaults to the constructor's docstring
):
    "Register a constructor with its CSS and behaviors, returning the constructor"
    if name is None: return partial(hypelt, sel=sel, css=css, on=on, layer=layer, doc=doc)
    if isinstance(name, str):
        fn = getattr(fasthtml.components, name)
        fn.__name__,fn.__doc__ = name,doc
        if sel is None: sel = fn().tag
    else:
        fn,name = name,name.__name__
        if sel is None: raise ValueError('A handwritten constructor needs sel')
        if doc is None: doc = fn.__doc__
    Rules(name, css, on, sel, fn, layer, doc or '')
    return fn

As an example component, let's make a dismissible notice — a box announcing something like "changes saved", with a button to close it. HTML has no element for that, so it gets a custom tag, `notice-box`, and its behavior is that a click on any child marked `dismiss` removes the notice:

In [ ]:
notice_css = r"""
notice-box {
  display: block;
  padding: .75rem 1rem;
  background: #f4f4f6;
  border: 1px solid #ccc;
  border-radius: 8px;
}
"""
NoticeBox = hypelt('NoticeBox', css=notice_css, on={('& [dismiss]', 'click'): r"el.closest('notice-box').remove()"},
    layer='components', doc='A dismissible notice.')
to_xml(NoticeBox('Changes saved.', Button('Dismiss', dismiss=True)))

'<notice-box>Changes saved.<button dismiss>Dismiss</button></notice-box>'

FastHTML generated the constructor from the CamelCase name, and since calling it with no arguments gives an empty element, `hypelt` read the root selector straight off the tag. The definition is a separate object in the registry:

In [ ]:
test_eq(NoticeBox().tag, _registry['NoticeBox'].sel)
_registry['NoticeBox']

NoticeBox [components] notice-box: A dismissible notice.

The second case is a handwritten constructor. Consider a labelled field. It wants native `label` and `input` elements, with the input wrapped in its label so the two are associated without an id. Our function, `Field`, needs arguments, so there's no empty call to read a selector from — we mark the label with a `field` attribute and supply it as `sel`, using the decorator form:

In [ ]:
field_css = r"""
label[field] {
  display: grid;
  gap: .5rem;
}
"""

@hypelt(sel='label[field]', css=field_css)
def Field(label, **attrs):
    "A labelled input"
    return Label(label, Input(**attrs), field=True)

to_xml(Field('Email', name='email', type='email'))

'<label field>Email  <input name="email" type="email">\n</label>'

## Overrides and redefinition

There are two ways to change how a component looks. An override registers separate CSS in the `app` layer, leaving the component's entry alone; redefinition registers the same name again, and the new entry replaces the old one completely.

Override example: a developer wants rounder notices for their application. In the stylesheet, the component's CSS sits in `components` and the override after it in `app`, so it wins:

In [ ]:
Rules('app-notices', 'notice-box { border-radius: 1rem; }', doc='Rounder notices in this application.')
print(build_css())

@layer reset, tokens, base, layout, components, app, utilities;

@layer components {
/* NoticeBox */
notice-box {
  display: block;
  padding: .75rem 1rem;
  background: #f4f4f6;
  border: 1px solid #ccc;
  border-radius: 8px;
}
}

@layer app {
/* app-page */
main { max-width: 65ch; margin-inline: auto; padding: 1rem; }
/* Field */
label[field] {
  display: grid;
  gap: .5rem;
}
/* app-notices */
notice-box { border-radius: 1rem; }
}



Redefinition example: redeclaring the NoticeBox hyperelement with custom CSS

In [ ]:
del _registry['app-notices']
NoticeBox = hypelt('NoticeBox', css=notice_css.replace('8px', '1rem'), layer='components', doc='A persistent notice.')
print(build_css())

@layer reset, tokens, base, layout, components, app, utilities;

@layer components {
/* NoticeBox */
notice-box {
  display: block;
  padding: .75rem 1rem;
  background: #f4f4f6;
  border: 1px solid #ccc;
  border-radius: 1rem;
}
}

@layer app {
/* app-page */
main { max-width: 65ch; margin-inline: auto; padding: 1rem; }
/* Field */
label[field] {
  display: grid;
  gap: .5rem;
}
}



Note that in the redefinition, we left out `on`, so it now has no behaviors.

In [ ]:
test_eq(_registry['NoticeBox'].on, [])

## Vocabulary

The registry doubles as an inventory: every entry was registered the same way, so listing them lists the whole vocabulary, the application's entries beside the built-ins:

In [ ]:
class Vocab(list):
    def __repr__(self): return '\n'.join(map(repr, self))

def vocab():
    "Every registered entry, one line each"
    return Vocab(_registry.values())

In [ ]:
vocab()

app-page [app]: A centered single-column page.
copy [app] [copy]: A control carrying `copy` copies its value, or its text, to the clipboard.
NoticeBox [components] notice-box: A persistent notice.
Field [app] label[field] Field(label, **attrs): A labelled input

Only `Field` shows a signature. A handwritten constructor's calling convention lives there, while a generated one takes children and attributes like every FastHTML tag, so its name says enough.

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()